# Universal Data Cleaner 2.0

A reference workflow for cleaning tabular data **without leaking test-set
information** and without breaking at inference time.

This notebook uses the reusable `data_cleaner.py` module in this folder. The
important pattern is:

1. inspect the raw data;
2. split into train/test **before** cleaning;
3. fit the cleaner on training data only;
4. transform train, test, and future batches with the learned rules;
5. persist the fitted cleaner with the model artifact.

## 1. Environment and imports

The project is managed with `uv`. If you are running this notebook outside the
project environment, install the dependencies first:

```bash
uv sync
```

In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data_cleaner.py").exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / "data_cleaner.py").exists():
            PROJECT_ROOT = candidate
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_cleaner import (
    CleanerConfig,
    DataCleaner,
    __version__,
    assert_frame_clean,
    dataset_summary,
    drop_duplicate_rows,
    profile_dataframe,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(name)s: %(message)s",
)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

print(f"data_cleaner version: {__version__}")
print(f"project root: {PROJECT_ROOT}")
print(f"pandas: {pd.__version__} | numpy: {np.__version__}")

## 2. Build a realistic dirty dataset

The sample data intentionally contains the failures that show up in real
tabular datasets:

- missing numbers, categories, booleans, and dates;
- numeric values stored as dirty strings (`"unknown"`);
- extreme numeric outliers;
- an invalid date string;
- exact duplicate rows;
- a constant column;
- an ID column that should not be fed to the model.

In [ ]:
rng = np.random.default_rng(42)
n_rows = 360

age = rng.integers(21, 60, n_rows)
experience = np.clip(age - rng.integers(18, 25, n_rows), 0, 40)
salary = 35_000 + experience * 2_600 + rng.normal(0, 9_000, n_rows)
department = rng.choice(
    ["Engineering", "Sales", "Support", "Finance"],
    n_rows,
    p=[0.40, 0.25, 0.20, 0.15],
)
country = rng.choice(["US", "UK", "DE"], n_rows, p=[0.50, 0.30, 0.20])
contract = rng.choice(
    ["Full-time", "Part-time", "Contract"],
    n_rows,
    p=[0.70, 0.20, 0.10],
)
signup = pd.Timestamp("2022-01-01") + pd.to_timedelta(
    rng.integers(0, 900, n_rows), unit="D"
)
is_active = pd.array(rng.random(n_rows) > 0.18, dtype="boolean")

department_effect = (
    pd.Series(department)
    .map({"Engineering": 4.0, "Sales": 2.0, "Support": 0.0, "Finance": 1.5})
    .to_numpy()
)
performance = (
    52
    + 0.22 * (age - 38)
    + 0.16 * (experience - 10)
    + 0.00012 * (salary - 70_000)
    + department_effect
    + rng.normal(0, 4.0, n_rows)
)
performance = np.clip(performance, 0, 100).round(1)

df = pd.DataFrame(
    {
        "EmployeeId": [f"E{i:04d}" for i in range(n_rows)],
        "Age": age,
        "Salary": salary.round(2),
        "Experience": experience,
        "Department": department,
        "Country": country,
        "ContractType": contract,
        "SignupDate": signup.strftime("%Y-%m-%d"),
        "IsActive": is_active,
        "DataSource": "crm_v3",
        "PerformanceScore": performance,
    }
)


def sample_index(count: int, seed: int) -> pd.Index:
    return df.sample(count, random_state=seed).index


df.loc[sample_index(14, 1), "Age"] = np.nan
df.loc[sample_index(12, 2), "Experience"] = np.nan
df.loc[sample_index(11, 3), "Department"] = None
df.loc[sample_index(9, 4), "Country"] = None

df["Salary"] = df["Salary"].astype(object)
df.loc[sample_index(8, 5), "Salary"] = "unknown"
df.loc[sample_index(4, 6), "Salary"] = "5000000"

df.loc[sample_index(7, 7), "SignupDate"] = "not-a-date"
df.loc[sample_index(10, 8), "IsActive"] = pd.NA

# Add exact duplicates, then shuffle so they are not all at the bottom.
df = pd.concat([df, df.sample(10, random_state=11)], ignore_index=True)
df = df.sample(frac=1, random_state=12).reset_index(drop=True)

print(f"raw dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

In [ ]:
audit = profile_dataframe(df)
display(audit)

summary = pd.DataFrame([dataset_summary(df)])
display(summary)

The profile already shows the important issue: `Salary` is an `object` column
because of the dirty strings, even though it is semantically numeric. That is
why the configuration below explicitly marks it as numeric.

## 3. Split before cleaning

Duplicate removal is a data-quality step, so it happens before the split. Every
**learned** cleaning parameter (medians, modes, IQR bounds, schema) is fitted
only on `X_train`.

In [ ]:
rows_before_dedup = len(df)
df, duplicates_removed = drop_duplicate_rows(df)
print(
    f"removed {duplicates_removed} exact duplicate rows ({rows_before_dedup} → {len(df)})"
)

stratify_key = df["Department"].fillna("Missing")
train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    random_state=42,
    stratify=stratify_key,
)

target = "PerformanceScore"
id_column = "EmployeeId"

X_train = train_df.drop(columns=[id_column, target])
y_train = train_df[target]
X_test = test_df.drop(columns=[id_column, target])
y_test = test_df[target]

# Simulate drift that can appear in a real inference batch.
X_test = X_test.copy()
X_test.loc[X_test.index[0], "Department"] = "Legal"  # unseen category
X_test.loc[X_test.index[1], "Salary"] = "9999999"  # extreme unseen value
X_test["UnexpectedSource"] = "mobile_app"  # unexpected column

print(f"train: {X_train.shape} | test: {X_test.shape}")
print(f"target range: {y_train.min():.1f} to {y_train.max():.1f}")

## 4. Fit the cleaner on training data only

`CleanerConfig` makes the implicit decisions explicit:

- `Salary` must be numeric despite arriving as dirty strings;
- `SignupDate` is expanded into calendar features rather than treated as a
  high-cardinality category;
- missing indicators preserve the information that a value was absent;
- constant columns are removed;
- unseen categories are kept so the downstream encoder can decide what to do.

The `cleaning_summary()` and `outlier_summary()` methods expose exactly what was
learned, which is useful for review and debugging.

In [ ]:
config = CleanerConfig(
    numeric_columns=("Salary",),
    datetime_columns=("SignupDate",),
    datetime_features=("year", "month", "day", "dayofweek", "is_weekend"),
    numeric_imputation="median",
    categorical_imputation="mode",
    cap_outliers=True,
    outlier_multiplier=1.5,
    add_missing_indicators=True,
    drop_constant_columns=True,
    unseen_category="keep",
)

cleaner = DataCleaner(config)
cleaner.fit(X_train)

print(f"fit rows: {cleaner.n_samples_fit_}")
print("unexpected column handling: ignored at transform time")
display(cleaner.cleaning_summary())

outlier_view = cleaner.outlier_summary()
outlier_view = outlier_view[
    outlier_view["column"].isin(["Age", "Salary", "Experience"])
]
display(outlier_view)

## 5. Transform train and test with the same learned rules

`transform()` never re-learns anything. In particular, the test set does not
contribute to the median, IQR bounds, or category vocabulary.

The test batch deliberately contains:

- a `Legal` department that did not exist in training;
- a `9,999,999` salary;
- an `UnexpectedSource` column.

The cleaner caps the salary, keeps the unseen category, ignores the unexpected
column, and returns the same schema as training.

In [ ]:
X_train_clean = cleaner.transform(X_train)
X_test_clean = cleaner.transform(X_test)

assert_frame_clean(X_train_clean)
assert_frame_clean(X_test_clean)

print(f"clean train: {X_train_clean.shape}")
print(f"clean test:  {X_test_clean.shape}")
print("The test batch was transformed without re-fitting.")


def frame_stats(frame: pd.DataFrame, stage: str) -> dict[str, object]:
    return {
        "stage": stage,
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing_cells": int(frame.isna().sum().sum()),
        "memory_mb": round(frame.memory_usage(deep=True).sum() / 1024**2, 2),
    }


comparison = pd.DataFrame(
    [
        frame_stats(X_train, "train raw"),
        frame_stats(X_train_clean, "train clean"),
        frame_stats(X_test, "test raw"),
        frame_stats(X_test_clean, "test clean"),
    ]
)
display(comparison)
display(X_train_clean.head())

In [ ]:
raw_columns = list(X_train.columns)
clean_columns = list(X_train_clean.columns)

print(f"raw columns:   {len(raw_columns)}")
print(f"clean columns: {len(clean_columns)}")
print()
print("Cleaned schema:")
for name in clean_columns:
    print(f"  - {name}: {X_train_clean[name].dtype}")

assert list(X_test_clean.columns) == clean_columns
assert "SignupDate" not in X_train_clean.columns
assert "DataSource" not in X_train_clean.columns
assert "UnexpectedSource" not in X_test_clean.columns
assert X_test_clean["Department"].eq("Legal").any()
assert X_test_clean["Salary"].max() < 1_000_000
print()
print("Schema, drift handling, and outlier caps verified.")

## 6. Persist and reload the fitted cleaner

The cleaner is part of the model artifact. Saving it means batch inference does
not need to re-read training data or re-compute medians.

In [ ]:
artifact_path = PROJECT_ROOT / "artifacts" / "data_cleaner.joblib"
cleaner.save(artifact_path)

restored = DataCleaner.load(artifact_path)
pd.testing.assert_frame_equal(
    X_test_clean,
    restored.transform(X_test),
)

print(f"saved artifact: {artifact_path}")
print(f"artifact size: {artifact_path.stat().st_size / 1024:.1f} KiB")
print("Reloaded cleaner produces an identical test transform.")

## 7. End-to-end scikit-learn Pipeline

The cleaner can sit directly inside a `Pipeline`. This is the recommended
production shape because the same object handles fitting, validation, test, and
future inference batches.

In [ ]:
numeric_features = [
    column for column in cleaner.numeric_columns_ if column in X_train_clean.columns
]
boolean_features = [
    column for column in cleaner.boolean_columns_ if column in X_train_clean.columns
]
categorical_features = [
    column for column in cleaner.categorical_columns_ if column in X_train_clean.columns
]

print("numeric:", numeric_features)
print("boolean:", boolean_features)
print("categorical:", categorical_features)

feature_pipeline = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("boolean", "passthrough", boolean_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
    remainder="drop",
)

model_pipeline = Pipeline(
    steps=[
        ("cleaner", DataCleaner(config)),
        ("features", feature_pipeline),
        ("model", Ridge(alpha=1.0)),
    ]
)

model_pipeline.fit(X_train, y_train)
predictions = model_pipeline.predict(X_test)

print(f"test R²: {r2_score(y_test, predictions):.3f}")
print(f"predictions: {len(predictions)} rows")

## 8. Production guardrails

These checks are intentionally small and executable. They catch the failures
that usually slip through notebook-only cleaning:

- output still has missing or infinite values;
- the input frame was mutated;
- the cleaner re-learned from a test batch;
- required columns disappeared;
- unexpected columns leaked into the model;
- the object no longer works with `sklearn.clone`.

In [ ]:
# 1. Clean output has no missing/infinite values.
assert_frame_clean(X_train_clean)

# 2. transform() does not mutate the caller's DataFrame.
input_snapshot = X_test.copy(deep=True)
_ = cleaner.transform(X_test)
pd.testing.assert_frame_equal(X_test, input_snapshot)

# 3. A test batch cannot change learned training parameters.
salary_fill_before = cleaner.fill_values_["Salary"]
probe = X_test.copy(deep=True)
probe["Salary"] = "9999999"
_ = cleaner.transform(probe)
assert cleaner.fill_values_["Salary"] == salary_fill_before

# 4. Missing required columns fail loudly.
try:
    cleaner.transform(X_train.drop(columns=["Age"]))
except ValueError as exc:
    assert "missing required columns" in str(exc).lower()
else:
    raise AssertionError("missing column did not raise ValueError")

# 5. Unexpected columns are ignored, not passed to the model.
assert "UnexpectedSource" not in cleaner.transform(X_test).columns

# 6. The cleaner is cloneable by scikit-learn.
clone(cleaner)

print("PASS: clean output")
print("PASS: input not mutated")
print("PASS: no test-set leakage")
print("PASS: missing-column guard")
print("PASS: unexpected-column guard")
print("PASS: sklearn clone")

## 9. Batch inference

A batch-scoring function only needs the saved artifact and the raw input batch.
It does not need the training frame.

In [ ]:
def clean_batch(
    batch: pd.DataFrame,
    artifact: str | Path = artifact_path,
) -> pd.DataFrame:
    """Load the persisted cleaner and apply it to a raw batch."""
    cleaner_from_disk = DataCleaner.load(artifact)
    cleaned = cleaner_from_disk.transform(batch)
    assert_frame_clean(cleaned)
    return cleaned


batch = X_test.head(3).copy()
batch.loc[batch.index[0], "Salary"] = "not-a-number"
batch.loc[batch.index[1], "Department"] = "NewDepartment"

cleaned_batch = clean_batch(batch)
display(cleaned_batch)

## 10. What to change for your own dataset

1. Drop IDs, free-text notes, and the target from `X` before fitting the cleaner.
2. Put dirty numeric columns in `CleanerConfig.numeric_columns` (for example,
   currency strings or numbers mixed with `"unknown"`).
3. Put date-like columns in `datetime_columns`; they will be replaced by calendar
   features.
4. Decide whether unseen categories should be kept (`"keep"`) or replaced
   (`"fill"`).
5. Fit on `X_train` after the split, never on the full dataset.
6. Save the cleaner with the model and load the pair together in production.
7. Re-run the guardrail cell whenever the schema or data source changes.

## Production checklist

- [x] split before cleaning
- [x] train-only medians, modes, IQR bounds, and category vocabulary
- [x] stable output schema and column order
- [x] no mutation of the caller's data
- [x] explicit handling for unseen categories and unexpected columns
- [x] persisted artifact with version metadata
- [x] sklearn `Pipeline` integration
- [x] executable tests in `tests/test_data_cleaner.py`